In [1]:
%pip install -Uq "unstructured[all-docs]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -Uq langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -Uq langchain-community langchain

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -Uq python_dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install langchain_huggingface

  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_huggingface-1.2.1-py3-none-any.whl (30 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_community.embeddings import SentenceTransformerEmbeddings
from dotenv import load_dotenv

load_dotenv()

c:\Projects\udemy\aws-courses\generativeAI\RAG_Practice\RAG\MultiModalRag\rag-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Step 1: Partitioning PDF into atomic Elements

In [2]:
def partition_document(file_path:str) :
    """Extract elements from PDF using unstructured"""
    print(f" Partitioning Document : {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy="hi_res", # using hi_res instead of basic_res. It  is slower but accurate
        infer_table_structure=True, # keep tables as structured HTML , not jumbled Text
        extract_image_block_types = ["Image"], # Grab images found in the pdf
        extract_image_block_to_payload = True, # convert image to base64
    )

    print(f"Extracted {len(elements)} of elements")
    return elements

file_path = "./docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

 Partitioning Document : ./docs/attention-is-all-you-need.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 5538.82it/s]


Extracted 215 of elements


In [3]:
# lets see the different types of elements extracted
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [4]:
# to get the contents of the element say we need content of the element 15
elements[15].to_dict()

{'type': 'Title',
 'element_id': 'fe5c284d230e37bff8d89e6b84355273',
 'text': 'Attention Is All You Need',
 'metadata': {'detection_class_prob': 0.4731905460357666,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(586.5582275390625),
     np.float64(412.293385)),
    (np.float64(586.5582275390625), np.float64(461.3277282714844)),
    (np.float64(1110.9853515625), np.float64(461.3277282714844)),
    (np.float64(1110.9853515625), np.float64(412.293385))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-10T13:50:37',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf',
  'parent_id': 'c549ee7dedfbebc5aea3efcdb29c64c3'}}

In [5]:
elements[14].to_dict()

{'type': 'NarrativeText',
 'element_id': '2dfda82d45d74a6d102b425fa2e28323',
 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.',
 'metadata': {'detection_class_prob': 0.8745211362838745,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(336.02191162109375),
     np.float64(201.6392311111111)),
    (np.float64(336.02191162109375), np.float64(313.4625244140625)),
    (np.float64(1365.7364501953125), np.float64(313.4625244140625)),
    (np.float64(1365.7364501953125), np.float64(201.6392311111111))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-10T13:50:37',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf',
  'parent_id': 'c549ee7dedfbebc5aea3efcdb29c64c3'}}

In [6]:
elements[1].to_dict()

{'type': 'UncategorizedText',
 'element_id': '9fe7dc93a917a205e3af37648566668a',
 'text': '2023',
 'metadata': {'coordinates': {'points': ((np.float64(52.0), np.float64(599.0)),
    (np.float64(52.0), np.float64(704.0)),
    (np.float64(89.0), np.float64(704.0)),
    (np.float64(89.0), np.float64(599.0))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-10T13:50:37',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf'}}

In [7]:
# get all the image elements
images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images")

Found 7 images


In [8]:
# tabels
tables = [element for element in elements if element.category == "Table"]
print(f"Found {len(tables)} tables")

Found 4 tables


In [9]:
# incase of image of table elements we dont use the "text attribute of the elements but instead
# for images we use raw_text and for tables we use "text-as-html" so we get the actual image and table instead of just text inside them which will be scrambled

Step 2: Chunking elements by Title

In [10]:
# Now using Unstructred we have paritioned the pdf documents into atomic elements
# we use these atomic elements and group them into chunks. We group them using title

def create_chunks_by_title(elements):
    """create intelligent chunks using title-based strategy"""
    print(f"Creating chunks.....")
    chunks = chunk_by_title(
        elements,
        max_characters=3000, # never exceed 3000 characters per chunk
        new_after_n_chars=2400, # try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [11]:
chunks = create_chunks_by_title(elements)

Creating chunks.....
Created 25 chunks


In [12]:
chunks

In [13]:
# lets get the type of the chunks
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [14]:
chunks[15].to_dict()

{'type': 'CompositeElement',
 'element_id': '5acb5aee-2944-4fb8-bf76-7c9c5d89432e',
 'text': '5.2 Hardware and Schedule\n\nWe trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps (3.5 days).\n\n5.3 Optimizer\n\nWe used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning rate over the course of training, according to the formula:\n\nlrate = d−0.5 model · min(step_num−0.5,step_num · warmup_steps−1.5) (3)\n\nThis corresponds to increasing the learning rate linearly for the first warmup_steps training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used warmup_steps = 4000.',
 'm

In [15]:
# to see all the elements inside the chunk 
chunks[4].metadata.orig_elements

In [16]:
chunks[4].metadata.orig_elements[3].to_dict()

{'type': 'Image',
 'element_id': '6f5f5a92fc9fea5732681d6b267bf0a8',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention Add & Norm Masked Multi-Head Attention Add & Norm Feed Forward Add & Norm Multi-Head Attention Nx Positional Encoding O° Positional @ OY Encoding Input Output Embedding Embedding Inputs Outputs (shifted right)',
 'metadata': {'coordinates': {'points': ((np.float64(545.9972222222221),
     np.float64(200.00555555555542)),
    (np.float64(545.9972222222221), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(200.00555555555542))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-10T13:50:37',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'attention-

Step 3: summerize the contents of chunks using LLM and vectorize them

In [17]:
# We are going to summerize th econtents of the chunks , this we are doing because me might have a table so we grab that text as HTML and if there is image we grab base64 of the image
# so first we try to seperate the content types and get their respective format as discussed above

def separate_content_types(chunk):
    """Analyze what types of contents are in the chunk"""
    content_data = {
        'text' : chunk.text,
        'tables' : [],
        'images' : [],
        'types' : ['text']
    }

    # check for tables and images in the original elements
    if hasattr(chunk,'metadata') and hasattr(chunk.metadata,'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # if element is a table extract the data in html format
            if element_type == 'Table':
                content_data['types'].append('table')
                # get html format using getattr(object, attribute, default)
                table_html = getattr(element.metadata,'text_as_html', element.text)
                content_data['tables'].append(table_html)
            elif element_type == "Image":
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [41]:
# we will use Hugging Face end Point for the LLM model
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from huggingface_hub import InferenceClient
import os
from langchain_core.messages import HumanMessage , SystemMessage

# now for the chunks we got , we create a summary using LLM model, which we use as langchain documents in the vector db
def create_ai_enhanced_summary(text: str, tables: List[str], images : List[str])->str:
    """create AI enhanced summary for mixed content"""
    try:

        hf_token = os.getenv("HF-TOKEN")
        client = InferenceClient(api_key=hf_token)
        
        image_descriptions = []
        
        # STEP 1: Turn images into text descriptions (Very stable on free tier)
        # Using BLIP: a tiny, fast, and always-free model
        for img_b64 in images[:2]: # limit to 2 for speed
            try:
                caption = client.image_to_text(img_b64, model="Salesforce/blip-image-captioning-base")
                image_descriptions.append(caption.generated_text)
            except:
                image_descriptions.append("[Image analysis skipped]")

        # STEP 2: Combine everything into a text prompt
        final_prompt = f"""Summarize this document chunk for a searchable database.
        
        TEXT CONTENT: {text}
        TABLE DATA: {str(tables)}
        IMAGE DESCRIPTIONS: {", ".join(image_descriptions)}
        
        Provide a concise summary of the key facts and topics."""

        # STEP 3: Use a highly-supported free text model
        # Llama 3.1 8B is almost always available on the free serverless tier
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[{"role": "user", "content": final_prompt}],
            max_tokens=500
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f" AI Summarization Failed : {e}")

        summary = f"{text[:300]}...."
        if tables:
            summary += f"[contains {len(tables)} tables]"
        if images:
            summary += f"[Contains {len(images)} images]"
        return summary

In [42]:
def summarize_chunks(chunks):
    """Process all the chunks with AI summaries"""
    print("Processign Chunks with AI summaries")

    langchain_documents = []
    total_chunks = len(chunks)

    for i , chunk in enumerate(chunks):
        current_chunk = i+1
        print(f"Processing chunk {current_chunk}/{total_chunks}")

        # analyze chunk content
        content_data = separate_content_types(chunk)

        # debug prints
        print(f"Types found in the chunk : {content_data['types']}")
        print(f"Tables : {len(content_data['tables'])}, images : {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f"Create AI summary for mixed content")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f" AI summary created successfully")
                print(f"Enhanced Content preview{enhanced_content[:200]}")
            except Exception as e:
                print(f" AI summary failed using Raw Text")
                enhanced_content = content_data["text"]

        else:
            print(f"Using Raw Text (no tables/images)")
            enhanced_content = content_data["text"]

        doc = Document(
            page_content = enhanced_content,
            metadata = {
                "original_content" : json.dumps({
                    "raw_text":content_data['text'],
                    "tables_html":content_data['tables'],
                    "image_base_64":content_data["images"]
                })
            }
        )

        langchain_documents.append(doc)
    
    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents

processed_chunks = summarize_chunks(chunks)

Processign Chunks with AI summaries
Processing chunk 1/25
Types found in the chunk : ['text']
Tables : 0, images : 0
Using Raw Text (no tables/images)
Processing chunk 2/25
Types found in the chunk : ['text']
Tables : 0, images : 0
Using Raw Text (no tables/images)
Processing chunk 3/25
Types found in the chunk : ['text']
Tables : 0, images : 0
Using Raw Text (no tables/images)
Processing chunk 4/25
Types found in the chunk : ['text']
Tables : 0, images : 0
Using Raw Text (no tables/images)
Processing chunk 5/25
Types found in the chunk : ['text', 'image']
Tables : 0, images : 1
Create AI summary for mixed content
 AI summary created successfully
Enhanced Content preview**Document Summary:**

* **Model Architecture:** Describes the general architecture of competitive neural sequence transduction models, which follow an encoder-decoder structure.
* **Key Components:**
Processing chunk 6/25
Types found in the chunk : ['text']
Tables : 0, images : 0
Using Raw Text (no tables/images)
Proce

In [43]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "3\\n\\n2023\\n\\n2\\n\\n0\\n\\n2\\n\\ng u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7\\n\\n1\\n\\n:\\n\\nv\\n\\narXiv\\n\\ni\\n\\nX\\n\\nr\\n\\na\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani\\u2217\\n\\nGoogle Brain\\n\\navaswani@google.com\\n\\nNoam Shazeer\\u2217 Google Brain noam@google.com\\n\\nNiki Parmar\\u2217 Google Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217\\n\\nGoogle Research usz@google.com\\n\\nLlion Jones\\u2217\\n\\nGoogle Research\\n\\nllion@google.com\\n\\nAidan N. Gomez\\u2217 \\u2020 University of Toronto aidan@cs.toronto.edu\\n\\n\\u0141ukasz Kaiser\\u2217 Google Brain lukaszkaiser@google.com", "tables_html": [], "image_base_64": []}'}, page_content='3\n\n2023\n\n2\n\n0\n\n2\n\ng u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7\n\n

In [45]:
%pip install sentence-transformers

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB 19.3 MB/s  0:00:00
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ------------------------------ --------- 6.8/8.9 MB 38.1 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 29.0 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]

In [48]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# with the summarized chunks available create a vector store
def create_vector_store(documents, persist_directory = "dbv1/chroma_db"):
    """create and persist vector db """
    print(f"Creatring the vector store in chromadb")

    embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
    print(f"------ Creating Vector Store------")
    vectorStore = Chroma.from_documents(
        documents = documents,
        embedding = embedding_model,
        persist_directory=persist_directory,
        collection_metadata = {"smsw:space":"cosine"}
    )
    print(f"---Finsihed Creating Vector Store-----")

    return vectorStore

db = create_vector_store(processed_chunks)

Creatring the vector store in chromadb


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5971.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


------ Creating Vector Store------
---Finsihed Creating Vector Store-----


In [50]:
# Lets test the rag
query = "How many attention heads does the transformer use , and what is the dimension of each head?"

retriever = db.as_retriever(search_kwargs = {"k":3})

chunks = retriever.invoke(query)

def generate_final_answer(chunks,query):
    """Generate the final answer using multimodal content"""
    try:
        #initialize the LLM
        # connect to the hugging face
        hf_token = os.getenv("HF-Token")
        client = InferenceClient(api_key = hf_token)

        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[{"role": "user", "content": message_content}],
            max_tokens=500
        )

        return response.choices[0].message.content
        
        
    except Exception as e:
        print(f" Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

Based on Document 1 and Document 3, the transformer uses multi-head attention in three different ways, including encoder-decoder attention, encoder self-attention, and decoder self-attention.

According to Document 3, the Transformer uses 8 parallel attention layers, or heads. This is mentioned in the text as: "In this work we employ h = 8 parallel attention layers, or heads."

As for the dimension of each head, Document 3 mentions that for each of the 8 parallel attention layers, or heads, the authors use dk = dv = dmodel/h = 64. This is also stated in the text: "dueto the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality."
